# ระบบผู้ช่วยปัญญาประดิษฐ์ (AI Assistant) ด้านกฎหมาย PDPA

สมุดบันทึก (Notebook) นี้ถูกจัดเตรียมขึ้นเพื่อดำเนินงาน **ขั้นตอนที่ 1: การประมวลผลข้อความและการแบ่งกลุ่มตามลำดับชั้น (Thai Text Processing & Hierarchy-Aware Chunking)**

## 1. การสร้างและเตรียมสภาพแวดล้อม (Environment)
เราได้สร้าง Conda Environment ที่ชื่อว่า `NSC` ผ่าน Miniconda เพื่อแยกไลบรารีของโปรเจกต์นี้ไม่ให้ปะปนกับโปรเจกต์อื่น โดยมีการติดตั้งแพ็กเกจที่จำเป็น เช่น `pdfplumber` และ `pythainlp`

**คำสั่งในการสร้าง Environment (รันใน Terminal):**
```bash
conda create -n NSC python=3.10 -y
conda activate NSC
```

In [2]:
# ติดตั้งแพ็กเกจที่จำเป็นสำหรับโปรเจกต์ (หากยังไม่ได้ติดตั้ง)
import sys
!{sys.executable} -m pip install pdfplumber pythainlp

## 2. ลอจิกสำหรับการสกัดข้อความ (Hierarchy-Aware Chunking)
โค้ดส่วนนี้จะอ่านข้อความจากไฟล์ `PDPA.pdf` และทำการตัดแบ่งเนื้อหาตามโครงสร้างของกฎหมาย (หมวด และ มาตรา) โดยมีการทำความสะอาดข้อความและตัดประโยคด้วยอัลกอริทึม CRF จาก PyThaiNLP

In [ ]:
import pdfplumber
import re
import json
from pythainlp.tokenize import word_tokenize, sent_tokenize

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

def clean_text(text):
    # ลบ Header/Footer ราชกิจจานุเบกษาทิ้ง เพื่อไม่ให้รบกวนข้อความกฎหมาย
    footer_pattern = re.compile(r'หน้า\s*[๐-๙0-9]+\s*เล่ม\s*[๐-๙0-9]+\s*ตอนที่\s*[๐-๙0-9]+\s*[ก-ฮ]?\s*ราชกิจจานุเบกษา\s*[๐-๙0-9]+\s*[ก-ฮา-์]+\s*[๐-๙0-9]+')
    text = footer_pattern.sub(' ', text)
    lines = text.split('\n')
    return [line.strip() for line in lines if line.strip()]

def process_thai_text(text):
    sentences = sent_tokenize(text, engine="crfcut")
    return " ".join([s.strip() for s in sentences if s.strip()])

def hierarchy_aware_chunking(lines, law_name="PDPA"):
    chunks = []
    current_chapter = "ไม่ระบุ"
    current_section = None
    current_chunk_lines = []
    
    # จับโครงสร้าง หมวด และ มาตรา
    chapter_pattern = re.compile(r'^หมวด(?:ที่)?\s*([๐-๙0-9]+)')
    section_pattern = re.compile(r'^มาตรา\s*([๐-๙0-9]+(?:\/[๐-๙0-9]+)?)')
    
    for line in lines:
        chapter_match = chapter_pattern.match(line)
        section_match = section_pattern.match(line)
        
        if chapter_match:
            current_chapter = chapter_match.group(1)
            continue
            
        if section_match:
            if current_section and current_chunk_lines:
                raw_text = " ".join(current_chunk_lines)
                chunks.append({
                    "metadata": {"law": law_name, "chapter": current_chapter, "section": current_section},
                    "text": process_thai_text(raw_text)
                })
            current_section = section_match.group(1)
            current_chunk_lines = [line]
        else:
            if current_section:
                current_chunk_lines.append(line)
                
    if current_section and current_chunk_lines:
        raw_text = " ".join(current_chunk_lines)
        chunks.append({
            "metadata": {"law": law_name, "chapter": current_chapter, "section": current_section},
            "text": process_thai_text(raw_text)
        })
        
    return chunks

## 3. ดำเนินการสร้าง JSON
เมื่อฟังก์ชันต่างๆ ถูกเตรียมพร้อมแล้ว ส่วนนี้จะเป็นการเริ่มกระบวนการสกัดข้อมูลและบันทึกไฟล์ผลลัพธ์เป็น `pdpa_structured_chunks.json`

In [ ]:
pdf_path = "PDPA.pdf"
output_json_path = "pdpa_structured_chunks.json"

raw_text = extract_text_from_pdf(pdf_path)
lines = clean_text(raw_text)
legal_chunks = hierarchy_aware_chunking(lines)

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(legal_chunks, f, ensure_ascii=False, indent=4)

print(f"✨ ทำการสร้าง Chunk สำเร็จทั้งหมด {len(legal_chunks)} มาตรา")

## ขั้นตอนที่ 2: การสร้างข้อมูลสังเคราะห์ (Synthetic Data Generation - SDG)

ขั้นตอนนี้เป็นการนำข้อมูลมาตรากฎหมายแต่ละ Chunk มาเป็นบริบทตั้งต้น (Context Injection) ให้กับ Teacher Model (เช่น GPT-4o) เพื่อสร้างชุดข้อมูลคู่คำถาม-คำตอบ (Instruction-Response Pairs) คุณภาพสูงสำหรับการทำ Fine-tuning

**กระบวนการให้เหตุผลทางกฎหมาย (Legal Chain-of-Thought - CoT)**
เราจะบังคับให้ LLM วิเคราะห์คำตอบตาม 4 ขั้นตอน:
1. ระบุบทบาทของนิติบุคคลในสถานการณ์ (ผู้ควบคุมข้อมูล / ผู้ประมวลผลข้อมูล)
2. จำแนกประเภทของข้อมูล (ข้อมูลส่วนบุคคลทั่วไป / ข้อมูลที่มีความอ่อนไหว)
3. ประเมินข้อยกเว้นภายใต้มาตราที่เกี่ยวข้อง
4. กำหนดคำแนะนำหรือคำสั่งในการปฏิบัติตามกฎหมายที่ชัดเจน

In [ ]:
# รันเซลล์นี้เพื่อติดตั้งไลบรารีของ OpenAI สำหรับการเรียกใช้งาน Teacher Model
import sys
!{sys.executable} -m pip install google-generativeai


In [ ]:
import json
import time
import os
import google.generativeai as genai

# กำหนด API Key
os.environ['GEMINI_API_KEY'] = 'YOUR_GEMINI_API_KEY' # <--- เปลี่ยนเป็น API Key ของคุณ
genai.configure(api_key=os.environ['GEMINI_API_KEY'])

# อัปเดตเป็นรุ่น gemini-3-flash-preview ตามที่คุณต้องการ
model = genai.GenerativeModel('gemini-3-flash-preview')

SYSTEM_PROMPT = """
คุณคือ "ผู้เชี่ยวชาญด้านกฎหมายคุ้มครองข้อมูลส่วนบุคคล (PDPA)"
เป้าหมายของคุณคือการสร้างชุดข้อมูลฝึกสอน (Training Data) คุณภาพสูงจำนวน 2-3 คู่ จากบริบท (Context) ที่กำหนดให้เท่านั้น ห้ามคิดข้อกำหนดกฎหมายขึ้นเองเด็ดขาด

[คำสั่งในการสร้างคำถาม (Instruction Generation)]
ให้สร้างคำถามหลากหลายรูปแบบ ครอบคลุมทั้ง:
- คำถามเชิงข้อเท็จจริง (Fact-based)
- คำถามเชิงสถานการณ์สมมติระดับองค์กร (Scenario-based)
- คำถามเชิงซับซ้อนที่ต้องอาศัยการตีความข้อยกเว้น (Complex Application)

[คำสั่งในการสร้างคำตอบ (Response Synthesis & Legal CoT)]
คำตอบของคุณ **ต้อง** แสดงกระบวนการคิดวิเคราะห์แบบนิรนัย (Deductive Reasoning) ตาม 4 ขั้นตอนนี้อย่างเคร่งครัด:
1. บทบาท: ระบุบทบาทของบุคคล/นิติบุคคล (เช่น ผู้ควบคุมข้อมูลส่วนบุคคล หรือ ผู้ประมวลผลข้อมูลส่วนบุคคล)
2. ประเภทข้อมูล: จำแนกประเภทของข้อมูล (ข้อมูลส่วนบุคคลทั่วไป หรือ ข้อมูลส่วนบุคคลที่มีความอ่อนไหว/Sensitive Data)
3. การประเมินกฎหมาย: ประเมินฐานความชอบด้วยกฎหมายและข้อยกเว้น ภายใต้มาตราที่เกี่ยวข้องในบริบท
4. คำแนะนำ: สรุปคำแนะนำหรือคำสั่งในการปฏิบัติตามกฎหมายที่ชัดเจน

[รูปแบบผลลัพธ์ (Output Format)]
ให้ส่งคืนผลลัพธ์เป็น JSON Array ล้วนๆ ในรูปแบบ:
{"qa_pairs": [
  {
    "instruction": "คำถาม...",
    "response": "1. บทบาท: ...\n2. ประเภทข้อมูล: ...\n3. การประเมินกฎหมาย: ...\n4. คำแนะนำ: ..."
  }
]}
"""

def generate_synthetic_data(chunk: dict, max_retries: int = 3) -> list:
    context = f"หมวด: {chunk['metadata']['chapter']}\nมาตรา: {chunk['metadata']['section']}\nเนื้อหากฎหมาย:\n{chunk['text']}"
    
    # ตั้งค่าให้ Gemini คืนค่าเป็น JSON
    generation_config = genai.GenerationConfig(
        response_mime_type="application/json",
        temperature=0.7
    )
    
    for attempt in range(max_retries):
        try:
            prompt = f"{SYSTEM_PROMPT}\n\nบริบทตั้งต้น:\n{context}\n\nจงสร้างข้อมูล QA Pairs ตามคำสั่ง"
            response = model.generate_content(
                prompt,
                generation_config=generation_config
            )
            
            # แกะข้อมูล JSON ที่โมเดลตอบกลับมา
            result_str = response.text
            parsed_json = json.loads(result_str)
            qa_pairs = parsed_json.get('qa_pairs', [])
            
            formatted_data = []
            for qa in qa_pairs:
                formatted_data.append({
                    "instruction": qa["instruction"],
                    "context": context,
                    "response": qa["response"]
                })
            return formatted_data
            
        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่มาตรา {chunk['metadata']['section']} (ครั้งที่ {attempt+1}): {e}")
            time.sleep(2)
            
    return []

def run_sdg_pipeline(input_json: str, output_jsonl: str, sample_size: int = None):
    with open(input_json, 'r', encoding='utf-8') as f:
        chunks = json.load(f)
        
    # จำกัดจำนวนในการสร้างเพื่อทดสอบ Pipeline
    target_chunks = chunks[:sample_size] if sample_size else chunks
    print(f"เริ่มกระบวนการ SDG จากข้อมูล {len(target_chunks)} มาตรา...")
    
    with open(output_jsonl, 'w', encoding='utf-8') as f:
        for i, chunk in enumerate(target_chunks):
            print(f"กำลังประมวลผล มาตรา {chunk['metadata']['section']} ({i+1}/{len(target_chunks)})...")
            generated_pairs = generate_synthetic_data(chunk)
            
            for pair in generated_pairs:
                f.write(json.dumps(pair, ensure_ascii=False) + '\n')
                
    print(f"\n✅ สร้างข้อมูลสังเคราะห์สำเร็จ บันทึกลงไฟล์ '{output_jsonl}'")


In [ ]:
import google.generativeai as genai
import os

# รันเซลล์นี้เพื่อดูรายชื่อโมเดลทั้งหมดที่คุณมีสิทธิ์ใช้งานผ่าน API Key นี้
genai.configure(api_key=os.environ.get('GEMINI_API_KEY'))
print("=== รายชื่อโมเดลที่รองรับ generateContent ===")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)


In [ ]:
# ==========================================
# ตัวอย่างการรัน Pipeline (กดรันเซลล์นี้เพื่อทำงาน)
# ==========================================
# หมายเหตุ: กรุณาใส่ OPENAI_API_KEY ในเซลล์ด้านบนก่อน
# แนะนำให้ลองรัน 5 มาตราแรกเพื่อตรวจสอบความถูกต้องก่อนรันทั้งหมด

run_sdg_pipeline("pdpa_structured_chunks.json", "pdpa_synthetic_data.jsonl", sample_size=5)